### ЗАДАЧА: Панель обработки проблемных доставок по паттерну `MVC`

Логистическая команда обрабатывает кейсы по проблемным доставкам.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `delay_hours` — задержка в часах;
- `status` — текущий статус;
- `operator` — сотрудник, который ведет кейс;
- `compensation_ready` — подготовлена ли компенсация;
- `resolution` — финальное решение.

Статусы кейса:
- `new`
- `investigating`
- `customer_contacted`
- `resolved`
- `escalated`

Бизнес-правила:
- нельзя создать кейс с уже существующим `case_id`;
- нельзя начать расследование без назначенного `operator`;
- перевести кейс в `customer_contacted` можно только из `investigating`;
- подготовить компенсацию можно только из `customer_contacted`;
- завершить кейс как `resolved` можно только из `customer_contacted`;
- финальный кейс (`resolved`, `escalated`) нельзя менять дальше;
- при `resolved` нужно записать `resolution` и оставить `compensation_ready` как есть;
- эскалировать можно только из `investigating` или `customer_contacted`.

In [1]:
from dataclasses import dataclass


initial_cases = [
    ("DL-100", "SHP-11", "FastBox", 6),
    ("DL-101", "SHP-12", "QuickShip", 18),
]

actions = [
    ("show",),
    ("investigate", "DL-100"),
    ("assign", "DL-100", "Olga"),
    ("investigate", "DL-100"),
    ("contact", "DL-100"),
    ("resolve", "DL-100", "refund_sent"),
    ("create", "DL-102", "SHP-13", "CityRun", 30),
    ("assign", "DL-102", "Max"),
    ("investigate", "DL-102"),
    ("escalate", "DL-102"),
    ("contact", "DL-102"),
    ("show",),
]


@dataclass
class DeliveryCase:
    case_id: str
    shipment_id: str
    courier: str
    delay_hours: int
    status: str = "new"
    operator: str | None = None
    compensation_ready: bool = False
    resolution: str | None = None


class DeliveryModel:
    final_statuses = {'resolved', 'escalated'}
    def __init__(self):
        self.cases = {}

    def _get_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError('Case not found')
        return self.cases[case_id]

    def add_case(self, case_id: str, shipment_id: str, courier: str, delay_hours: int) -> None:
        if case_id in self.cases:
            raise ValueError('Case already exists')
        self.cases[case_id] = DeliveryCase(case_id, shipment_id, courier, delay_hours)

    def assign_operator(self, case_id: str, operator: str) -> None:
        case = self._get_case(case_id)
        if case.status in self.final_statuses:
            raise ValueError('Cannot change cases with final statuses')
        case.operator = operator

    def start_investigation(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status in self.final_statuses:
            raise ValueError('Cannot change cases with final statuses')
        if case.operator is None:
            raise ValueError('Operator is required')
        case.status = "investigating"

    def contact_customer(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status != 'investigating':
            raise ValueError('"investigating" status is required')
        case.status = "customer_contacted"

    def prepare_compensation(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status != 'customer_contacted':
            raise ValueError('"customer_contacted" status is required')
        
        
        case.compensation_ready = True

    def resolve_case(self, case_id: str, resolution: str) -> None:
        case = self._get_case(case_id)
        if case.status != 'customer_contacted':
            raise ValueError('"customer_contacted" status is required')
        case.status = "resolved"
        case.resolution = resolution

    def escalate_case(self, case_id: str) -> None:
        case = self._get_case(case_id)
        if case.status not in {'investigating', 'customer_contacted'}:
            raise ValueError('can escalate the case only if status is "investigating" or "customer_contacted"')
        case.status = "escalated"

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.shipment_id} | {case.courier} | {case.delay_hours} | "
                f"{case.status} | {case.operator} | {case.compensation_ready} | {case.resolution}"
            )
        return rows


class DeliveryView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Delivery cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class DeliveryController:
    def __init__(self, model: DeliveryModel, view: DeliveryView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, shipment_id: str, courier: str, delay_hours: int) -> None:
        try:
            self.model.add_case(case_id, shipment_id, courier, delay_hours)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_operator(self, case_id: str, operator: str) -> None:
        try:
            self.model.assign_operator(case_id, operator)
            self.view.render_success(f"Operator assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_investigation(self, case_id: str) -> None:
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Case {case_id} moved to investigation")
        except ValueError as error:
            self.view.render_error(str(error))

    def contact_customer(self, case_id: str) -> None:
        try:
            self.model.contact_customer(case_id)
            self.view.render_success(f"Customer contacted for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def prepare_compensation(self, case_id: str) -> None:
        try:
            self.model.prepare_compensation(case_id)
            self.view.render_success(f"Compensation prepared for {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def resolve_case(self, case_id: str, resolution: str) -> None:
        try:
            self.model.resolve_case(case_id, resolution)
            self.view.render_success(f"Case {case_id} resolved")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())


model = DeliveryModel()
view = DeliveryView()
controller = DeliveryController(model, view)

for case_id, shipment_id, courier, delay_hours in initial_cases:
    model.add_case(case_id, shipment_id, courier, delay_hours)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, shipment_id, courier, delay_hours = action
        controller.create_case(case_id, shipment_id, courier, delay_hours)
    elif action[0] == "assign":
        _, case_id, operator = action
        controller.assign_operator(case_id, operator)
    elif action[0] == "investigate":
        _, case_id = action
        controller.start_investigation(case_id)
    elif action[0] == "contact":
        _, case_id = action
        controller.contact_customer(case_id)
    elif action[0] == "prepare":
        _, case_id = action
        controller.prepare_compensation(case_id)
    elif action[0] == "resolve":
        _, case_id, resolution = action
        controller.resolve_case(case_id, resolution)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()


Delivery cases:
DL-100 | SHP-11 | FastBox | 6 | new | None | False | None
DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
ERROR: Operator is required
SUCCESS: Operator assigned to DL-100
SUCCESS: Case DL-100 moved to investigation
SUCCESS: Customer contacted for DL-100
SUCCESS: Case DL-100 resolved
SUCCESS: Case DL-102 created
SUCCESS: Operator assigned to DL-102
SUCCESS: Case DL-102 moved to investigation
SUCCESS: Case DL-102 escalated
ERROR: "investigating" status is required
Delivery cases:
DL-100 | SHP-11 | FastBox | 6 | resolved | Olga | False | refund_sent
DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
DL-102 | SHP-13 | CityRun | 30 | escalated | Max | False | None

Финальное состояние:
Delivery cases:
DL-100 | SHP-11 | FastBox | 6 | resolved | Olga | False | refund_sent
DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
DL-102 | SHP-13 | CityRun | 30 | escalated | Max | False | None


In [ ]:
# Delivery cases:
# DL-100 | SHP-11 | FastBox | 6 | new | None | False | None
# DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
# ERROR: Operator is required
# SUCCESS: Operator assigned to DL-100
# SUCCESS: Case DL-100 moved to investigation
# SUCCESS: Customer contacted for DL-100
# SUCCESS: Case DL-100 resolved
# SUCCESS: Case DL-102 created
# SUCCESS: Operator assigned to DL-102
# SUCCESS: Case DL-102 moved to investigation
# SUCCESS: Case DL-102 escalated
# ERROR: Case is not investigating
# Delivery cases:
# DL-100 | SHP-11 | FastBox | 6 | resolved | Olga | False | refund_sent
# DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
# DL-102 | SHP-13 | CityRun | 30 | escalated | Max | False | escalated
#
# Финальное состояние:
# Delivery cases:
# DL-100 | SHP-11 | FastBox | 6 | resolved | Olga | False | refund_sent
# DL-101 | SHP-12 | QuickShip | 18 | new | None | False | None
# DL-102 | SHP-13 | CityRun | 30 | escalated | Max | False | escalated